In [ ]:
%gui qt
%load_ext autoreload
%autoreload 2

import hmt_v3 as hmt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Preprocessing and Formatting

In [ ]:
plt.rcParams.update({
    # --- text ---
    "font.size": 14,
    # "axes.titlesize": 16,
    # "axes.labelsize": 14,
    # "xtick.labelsize": 12,
    # "ytick.labelsize": 12,
    # "legend.fontsize": 14,
    # "figure.titlesize": 20,

    # --- lines & markers ---
    "lines.linewidth": 2.5,
    "lines.markersize": 8,
    "patch.linewidth": 1.5,      # bar/patch outlines
    "axes.linewidth": 1.5,       # axis spines

    # --- ticks ---
    "xtick.major.width": 1.5,
    "ytick.major.width": 1.5,
    "xtick.major.size": 6,
    "ytick.major.size": 6,

    # --- export quality ---
    "savefig.dpi": 300,
    "figure.dpi": 100,           # on-screen only; doesn't affect saved size
    "savefig.bbox": "tight",
})

## Simulated Data Generation

### Toy centroid model: me3/ac integration sweep

Synthetic centroid generator and spatial-distribution metrics, defined in `hmt_v3/postprocess.py` (independent of the real `me3_df` / `ac_df` and nucleus mask above). `integration_level` is a single knob (0 = fully independent CSR centroids per channel, 1 = every me3 centroid has a paired ac centroid) for testing centroid-based spatial-distribution metrics — colocalization fraction, cross pair-correlation, self-nonself contact ratio — against a known ground truth.

In [ ]:
# --- Generate balanced me3/ac centroid distributions across integration_level ---
rng = np.random.default_rng(0)
field_size_nm = 10000.0
coloc_radius_nm = 150.0
integration_levels = [0.0, 0.25, 0.5, 0.75, 1.0]

sweep = hmt.postprocess.generate_integration_sweep(
    integration_levels, n_domains=150, field_size_nm=field_size_nm, rng=rng)

fig, axes = plt.subplots(1, len(integration_levels), figsize=(4 * len(integration_levels), 4), sharey=True)
for row, ax in zip(sweep, axes):
    coloc = hmt.postprocess.colocalization_fraction(
        row["me3_seeds"][["x [nm]", "y [nm]"]].to_numpy(),
        row["ac_seeds"][["x [nm]", "y [nm]"]].to_numpy(), coloc_radius_nm)
    row["coloc_fraction"] = coloc

    ax.scatter(row["me3_seeds"]["x [nm]"], row["me3_seeds"]["y [nm]"], s=15, alpha=0.7, color="green", label="me3")
    ax.scatter(row["ac_seeds"]["x [nm]"], row["ac_seeds"]["y [nm]"], s=15, alpha=0.7, color="red", label="ac")
    ax.set_title(f"level={row['integration_level']}")
    ax.set_aspect("equal")

axes[0].legend(loc="upper right", fontsize=10)
plt.suptitle("Toy centroids (balanced)")
plt.tight_layout()
plt.show()

The balanced sweep above always gives me3 and ac the same total centroid count, so `me3_expected_ratio`/`ac_expected_ratio` ≈ 1 already and normalizing the self-nonself contact ratio barely changes anything — that's expected, not a bug, since normalization only corrects for a count imbalance that isn't present there. The demo below repeats the sweep with 2x as many me3 centroids as ac (`n_domains=300, n_domains_ac=150`) so the raw curves are dominated by that abundance imbalance while the normalized curves stay comparable to the balanced case.

In [ ]:
# --- Generate imbalanced me3/ac centroid distributions (300 me3 vs. 150 ac) ---
rng = np.random.default_rng(0)
integration_levels = [0.0, 0.25, 0.5, 0.75, 1.0]

imbalanced_sweep = hmt.postprocess.generate_integration_sweep(
    integration_levels, n_domains=300, n_domains_ac=150, field_size_nm=field_size_nm, rng=rng)

fig, axes = plt.subplots(1, len(integration_levels), figsize=(4 * len(integration_levels), 4), sharey=True)
for row, ax in zip(imbalanced_sweep, axes):
    ax.scatter(row["me3_seeds"]["x [nm]"], row["me3_seeds"]["y [nm]"], s=15, alpha=0.7, color="green",
               label=f"me3 (n={len(row['me3_seeds'])})")
    ax.scatter(row["ac_seeds"]["x [nm]"], row["ac_seeds"]["y [nm]"], s=15, alpha=0.7, color="red",
               label=f"ac (n={len(row['ac_seeds'])})")
    ax.set_title(f"level={row['integration_level']}")
    ax.set_aspect("equal")

axes[0].legend(loc="upper right", fontsize=10)
plt.suptitle("Toy centroids (imbalanced, 300 me3 vs. 150 ac)")
plt.tight_layout()
plt.show()

### Self-nonself contact ratio

`self_nonself_contact_ratio` and `plot_sncr_sweep` now live in `hmt_v3/postprocess.py` rather than the notebook. Plotted here on the balanced and imbalanced centroid distributions generated above.

In [ ]:
# --- SNCR: balanced distribution ---
hmt.postprocess.plot_sncr_sweep(
    sweep, field_size_nm, r_max=800.0, dr=25.0,
    title="Self-nonself contact ratio (balanced)")
plt.show()

In [ ]:
# --- SNCR: imbalanced distribution ---
hmt.postprocess.plot_sncr_sweep(
    imbalanced_sweep, field_size_nm, r_max=800.0, dr=25.0,
    title="Self-nonself contact ratio (imbalanced)")
plt.show()

### Ripley's K (bivariate cross-K/L)

`ripleys_k_cross` and `plot_ripley_k_sweep` (`hmt_v3/postprocess.py`) compute the standard bivariate Ripley's K function between me3 and ac centroids -- the disk-cumulative sibling of `cross_pair_correlation` (K is g's running integral), reported via the variance-stabilized `L(r) - r` (0 = CSR, >0 = attraction/integration at scale r, <0 = repulsion). Plotted on the same balanced and imbalanced distributions generated above.

In [ ]:
# --- Ripley's K/L: balanced distribution ---
hmt.postprocess.plot_ripley_k_sweep(
    sweep, field_size_nm, r_max=800.0, dr=25.0,
    title="Ripley's K/L (balanced)")
plt.show()

In [ ]:
# --- Ripley's K/L: imbalanced distribution ---
hmt.postprocess.plot_ripley_k_sweep(
    imbalanced_sweep, field_size_nm, r_max=800.0, dr=25.0,
    title="Ripley's K/L (imbalanced)")
plt.show()

### Graph-theory metrics: Delaunay assortativity & Friedman-Rafsky MST test

Two graph-based views of the same centroids (`hmt_v3/postprocess.py`), neither needing a chosen radius `r`:
- `delaunay_channel_mixing` / `plot_delaunay_sweep`: Delaunay triangulation over the pooled centroids, edges labelled homotypic/heterotypic; reports the raw heterotypic edge fraction and Newman's assortativity coefficient (chance-corrected, 0 = no mixing preference, -1 = maximally mixed, +1 = fully segregated).
- `friedman_rafsky_test` / `plot_mst_sweep`: minimum spanning tree over the pooled centroids, cross-type edges counted and compared to a label-permutation null (a proper two-sample statistical test for whether me3/ac come from the same spatial distribution).

In [ ]:
# --- Delaunay assortativity: balanced distribution ---
hmt.postprocess.plot_delaunay_sweep(sweep, title="Delaunay triangulation (balanced)")
plt.show()

In [ ]:
# --- Delaunay assortativity: imbalanced distribution ---
hmt.postprocess.plot_delaunay_sweep(imbalanced_sweep, title="Delaunay triangulation (imbalanced)")
plt.show()

In [ ]:
# --- Friedman-Rafsky MST test: balanced distribution ---
hmt.postprocess.plot_mst_sweep(sweep, n_permutations=500, rng=np.random.default_rng(0),
                               title="Minimum spanning tree (balanced)")
plt.show()

In [ ]:
# --- Friedman-Rafsky MST test: imbalanced distribution ---
hmt.postprocess.plot_mst_sweep(imbalanced_sweep, n_permutations=500, rng=np.random.default_rng(0),
                               title="Minimum spanning tree (imbalanced)")
plt.show()

### MST merge curve (H0 persistent homology)

A separate view of the same MST used above, from `mst_merge_curve` / `plot_merge_curve_sweep` (`hmt_v3/postprocess.py`). `friedman_rafsky_test` collapses the whole tree into one number (total cross-type edge count vs. a permutation null); this instead replays the MST's edges in increasing length order -- exactly the order components merge under a Vietoris-Rips filtration, i.e. 0-dimensional persistent homology -- and tracks the cumulative fraction of merges that are heterotypic as a function of merge radius. That answers a different question: not "how much cross-channel merging is there overall" but "at what length scale does it start."

In [ ]:
# --- MST merge curve: balanced distribution ---
hmt.postprocess.plot_merge_curve_sweep(sweep, r_max=800.0, title="MST merge curve (balanced)")
plt.show()

In [ ]:
# --- MST merge curve: imbalanced distribution ---
hmt.postprocess.plot_merge_curve_sweep(imbalanced_sweep, r_max=800.0, title="MST merge curve (imbalanced)")
plt.show()

### Separate-mesh overlap

Unlike `delaunay_channel_mixing` above (one triangulation pooling both channels), `mesh_overlap` / `plot_mesh_overlap_sweep` (`hmt_v3/postprocess.py`) build TWO independent triangulations -- one over me3 centroids only, one over ac centroids only -- and ask where the two meshes physically cross in space. This captures whether each channel's local connectivity *structure* threads through the other's, rather than whether individual domains sit next to each other. Includes an optional label-permutation significance test (same logic as Friedman-Rafsky, but rebuilding both triangulations each permutation instead of reshuffling one fixed tree, so it's slower -- kept to 100 permutations here).

In [ ]:
# --- Mesh overlap: balanced distribution ---
hmt.postprocess.plot_mesh_overlap_sweep(sweep, n_permutations=100, rng=np.random.default_rng(0),
                                        title="Delaunay mesh overlap (balanced)")
plt.show()

In [ ]:
# --- Mesh overlap: imbalanced distribution ---
hmt.postprocess.plot_mesh_overlap_sweep(imbalanced_sweep, n_permutations=100, rng=np.random.default_rng(0),
                                        title="Delaunay mesh overlap (imbalanced)")
plt.show()